In [5]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT_DIR = Path().absolute().parent.parent.parent.parent
PROJECT_ROOT_DIR

if str(PROJECT_ROOT_DIR.absolute()) not in sys.path:
    sys.path.append(str(PROJECT_ROOT_DIR.absolute()))

from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecCheckNan

from runF16.algorithm.models.sb3_model import PPOWithBCLoss
from runF16.algorithm.utils.sb3_eval_callback import MyEvalCallbackSTAT
from runF16.algorithm.utils.sb3_evaluate_policy import evaluate_policy_with_stat
from runF16.env.utils.sb3_utils import get_vec_env_dense_reward
from runF16.algorithm.utils.sb3_schedule import linear_schedule



In [6]:
BC_MODEL = "10hz_128_128_300epochs_loss_2"

In [7]:
ROLLOUT_PROCESS_NUM = 10
STEP_FREQUENCE = 10
MAX_SIMULATE_TIME = 40
GAMMA = 0.995
REWARD_SCALAR_B = 0.5
EXPERT_DATA_DIR = "10hz_10_5_5"
RL_LR_RATE = 3e-4
KL_WITH_BC_MODEL_COEF = 1e-3

In [8]:
vec_env = VecCheckNan(get_vec_env_dense_reward(**{
    "num_process": ROLLOUT_PROCESS_NUM, 
    "logger": None, 
    "step_frequence": STEP_FREQUENCE,
    "max_simulate_time": MAX_SIMULATE_TIME,
    "gamma": GAMMA,
    "reward_scale_factor_b": REWARD_SCALAR_B,  # GuideEnvDenseReward中使用，取值小于1效果相当于在数值上放大了奖励，大于1相当于缩小了奖励
    "available_targets_dir": PROJECT_ROOT_DIR / "my_logs_for_parallel" / EXPERT_DATA_DIR,
    "sample_target_noise_std": [0., 0., 0.],
}))
eval_callback_env = VecCheckNan(get_vec_env_dense_reward(**{
    "num_process": ROLLOUT_PROCESS_NUM, 
    "logger": None, 
    "step_frequence": STEP_FREQUENCE,
    "max_simulate_time": MAX_SIMULATE_TIME,
    "gamma": GAMMA,
    "reward_scale_factor_b": REWARD_SCALAR_B,  # GuideEnvDenseReward中使用，取值小于1效果相当于在数值上放大了奖励，大于1相当于缩小了奖励
    "available_targets_dir": PROJECT_ROOT_DIR / "my_logs_for_parallel" / EXPERT_DATA_DIR,
    "sample_target_noise_std": [0., 0., 0.],
}))

In [9]:
bc_policy_save_dir = PROJECT_ROOT_DIR / "runF16" / "algorithm" / "checkpoints_sb3" / "bc" / BC_MODEL
algo_ppo_for_kl_loss = PPOWithBCLoss.load(str((bc_policy_save_dir / "bc_checkpoint").absolute()))
algo_ppo_for_kl_loss.policy.set_training_mode(False)
algo_ppo = PPOWithBCLoss.load(
    str((bc_policy_save_dir / "bc_checkpoint").absolute()), 
    env=vec_env, 
    seed=1,
    custom_objects={
        "bc_trained_algo": algo_ppo_for_kl_loss,
        "learning_rate": linear_schedule(RL_LR_RATE),
    },
    kl_coef_with_bc=KL_WITH_BC_MODEL_COEF,
)

In [10]:
stat_success_dict, stat_fail_dict = {}, {}

eval_callback = MyEvalCallbackSTAT(
        eval_callback_env, 
        best_model_save_path=".",
        log_path=".", 
        eval_freq=100,  # 多少次env.step()评估一次，此处设置为1000，因为VecEnv有72个并行环境，所以实际相当于72*1000次step，评估一次
        n_eval_episodes=10,  # 每次评估使用多少条轨迹
        deterministic=True, 
        render=False,
        stat_success_dict=stat_success_dict,
        stat_fail_dict=stat_fail_dict
    )

algo_ppo.learn(total_timesteps=1000, callback=eval_callback)

/home/ucav/PythonProjects/618/runF16/algorithm/utils/sb3_evaluate_policy.py:215: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 130。target: (230.0, -10.0, -10.0)。expert steps: 129。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 280。target: (300.0, -15.0, 100.0)。expert steps: 279。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 283。target: (180.0, -10.0, -140.0)。expert steps: 263。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 318。target: (100.0, 15.0, 65.0)。expert steps: 315。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 319。target: (280.0, -20.0, -10.0)。expert steps: 179。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 388。target: (280.0, -25.0, -110.0)。expert steps: 342。
print,  reach_target_termination_based_on_angle_of_velocity_vector。 steps: 389。target: (280.0, -25.0, -120.0)。expert steps: 331。
print,  timeout_termination。 steps: 399。target: (120.0, 15.0, -150.0)。expert steps: 427。
print, print, 

In [11]:
stat_success_dict, stat_fail_dict

({'230_-10_-10': 1,
  '300_-15_100': 1,
  '180_-10_-140': 1,
  '100_15_65': 1,
  '280_-20_-10': 2,
  '280_-25_-110': 1,
  '280_-25_-120': 1,
  '230_0_25': 1,
  '140_15_15': 1,
  '160_5_15': 1,
  '130_10_50': 1,
  '210_10_-150': 1,
  '190_10_135': 1,
  '270_-15_-110': 1,
  '130_20_-115': 1,
  '250_-30_0': 1,
  '200_15_-35': 1,
  '220_-15_35': 1,
  '220_-5_120': 1,
  '180_0_145': 1,
  '230_-10_-165': 1,
  '140_15_-40': 1,
  '220_-5_-55': 1,
  '210_5_110': 1,
  '210_-10_80': 1,
  '290_-20_70': 1,
  '150_0_115': 1,
  '280_-25_80': 1,
  '250_-25_-70': 1,
  '180_10_35': 1,
  '200_5_55': 1,
  '100_40_-20': 1,
  '210_5_-145': 1,
  '180_-5_10': 1,
  '150_0_50': 1,
  '290_-20_80': 1,
  '170_0_-140': 1,
  '260_-5_-170': 1,
  '210_-5_45': 1,
  '250_-15_65': 1,
  '250_-20_50': 1,
  '180_15_110': 1,
  '270_-20_95': 1,
  '220_-15_150': 1,
  '290_-5_-160': 1,
  '140_0_70': 1,
  '210_-25_0': 1,
  '260_-5_10': 1,
  '140_15_40': 1,
  '270_5_15': 1,
  '230_-5_-70': 1,
  '250_10_155': 1,
  '180_-5_140': 1,